# ImageNet Object Ratio

Minimal version: one cumulative plot only.

- Uses full-image coverage rows only (`k = total / 3`).
- Sorts by `object ratio` from `fix_nonmask`.
- Uses three panels: `safe`, `unknown`, and `unsafe`.
- Each panel shows two curves only: `object` and `global`.


In [1]:
import os
import shutil
import subprocess
from pathlib import Path


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "results" / "vggnet16" / "all_k_vggnet16.csv").exists() and (candidate / "plots").exists():
            return candidate
    raise FileNotFoundError("Could not locate the XAIV project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
PLOT_DIR = PROJECT_ROOT / "plots" / "5.4.2_ratio"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = PLOT_DIR / ".cache"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_ROOT / "matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

texlive_root = Path("/opt/homebrew/Cellar/texlive")
tex_bin_candidates = [
    Path("/opt/homebrew/opt/texlive/bin"),
    *sorted(texlive_root.glob("*/bin"), reverse=True),
    Path("/opt/homebrew/bin"),
    Path("/Library/TeX/texbin"),
    Path("/usr/texbin"),
]
tex_lib_candidates = [
    Path("/opt/homebrew/opt/libpng/lib"),
    Path("/opt/homebrew/opt/texlive/lib"),
    *sorted(texlive_root.glob("*/lib"), reverse=True),
]

current_path = os.environ.get("PATH", "")
path_parts = current_path.split(":") if current_path else []
for texbin in tex_bin_candidates:
    if texbin.exists():
        texbin_str = str(texbin)
        if texbin_str not in path_parts:
            path_parts.insert(0, texbin_str)
os.environ["PATH"] = ":".join(path_parts)

current_dyld = os.environ.get("DYLD_LIBRARY_PATH", "")
dyld_parts = current_dyld.split(":") if current_dyld else []
for texlib in tex_lib_candidates:
    if texlib.exists():
        texlib_str = str(texlib)
        if texlib_str not in dyld_parts:
            dyld_parts.insert(0, texlib_str)
if dyld_parts:
    os.environ["DYLD_LIBRARY_PATH"] = ":".join(dyld_parts)

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
except ImportError:
    set_matplotlib_formats = None

FORCE_TEX = True
BASE_FONT_SIZE = 16
AXIS_LABEL_SIZE = 18
AXIS_TITLE_SIZE = 18
TICK_LABEL_SIZE = 16
LEGEND_FONT_SIZE = 13
QUANTILE_LABEL_SIZE = 11
DEFAULT_FIGSIZE = (15.6, 4.8)


def latex_ready():
    has_renderer = shutil.which("dvipng") is not None or shutil.which("dvisvgm") is not None
    return shutil.which("latex") is not None and has_renderer


def has_tex_package(package_name):
    if shutil.which("kpsewhich") is None:
        return False
    return subprocess.run(
        ["kpsewhich", f"{package_name}.sty"],
        capture_output=True,
        text=True,
        check=False,
    ).returncode == 0


def configure_plot_style(force_tex=FORCE_TEX):
    use_tex = force_tex and latex_ready()
    preamble = r"\usepackage{fontawesome5}" if use_tex and has_tex_package("fontawesome5") else ""

    if set_matplotlib_formats is not None:
        if use_tex and shutil.which("dvisvgm") is not None:
            set_matplotlib_formats("svg")
        else:
            set_matplotlib_formats("png")

    if force_tex and not use_tex:
        print("LaTeX was requested but no compatible renderer was found. Falling back to Matplotlib serif text.")

    mpl.rcParams.update({
        "text.usetex": use_tex,
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "STIXGeneral", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "text.latex.preamble": preamble,
        "font.size": BASE_FONT_SIZE,
        "axes.labelsize": AXIS_LABEL_SIZE,
        "axes.titlesize": AXIS_TITLE_SIZE,
        "xtick.labelsize": TICK_LABEL_SIZE,
        "ytick.labelsize": TICK_LABEL_SIZE,
        "legend.fontsize": LEGEND_FONT_SIZE,
        "axes.linewidth": 1.0,
        "grid.linewidth": 0.6,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    return use_tex


use_tex = configure_plot_style()

CSV_CANDIDATES = [
    PROJECT_ROOT / "results" / "vggnet16" / "all_k_vggnet16.csv",
]

for candidate in CSV_CANDIDATES:
    candidate = candidate.resolve()
    if candidate.exists():
        CSV_PATH = candidate
        break
else:
    checked = "\n".join(str(path.resolve()) for path in CSV_CANDIDATES)
    raise FileNotFoundError(f"Could not find the ImageNet CSV. Checked:\n{checked}")

df = pd.read_csv(CSV_PATH)

for col in ["eps", "k", "total", "segment_index", "num_changed"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["tag", "image", "model"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

df = df.dropna(subset=["total", "k", "num_changed"]).copy()
df = df[df["tag"].isin(["fix_nonmask", "global"])].copy()
df = df[df["total"].astype(int) == 150528].copy()
df["full_k"] = (df["total"] // 3).astype(int)
df = df[df["k"].astype(int) == df["full_k"]].copy()


def canon_result(value):
    text = str(value).lower()
    if "unsat" in text:
        return "safe"
    if "sat" in text:
        return "unsafe"
    return "unknown"


df["result_cat"] = df["result"].apply(canon_result)

key_global = [col for col in ["model", "image", "eps", "k", "total"] if col in df.columns]
key_object = [col for col in ["model", "image", "segment_index", "eps", "k", "total"] if col in df.columns]

object_df = (
    df[df["tag"] == "fix_nonmask"][key_object + ["num_changed", "result_cat"]]
    .rename(columns={"num_changed": "obj_changed", "result_cat": "obj_result"})
    .copy()
)

global_df = (
    df[df["tag"] == "global"][key_global + ["result_cat"]]
    .rename(columns={"result_cat": "glb_result"})
    .drop_duplicates(subset=key_global, keep="first")
    .copy()
)

paired = object_df.merge(global_df, on=key_global, how="left")
paired = paired.dropna(subset=["glb_result"]).copy()

if paired.empty:
    raise ValueError("No paired object/global rows found after filtering.")

paired["object_ratio"] = paired["obj_changed"] / paired["total"]
paired = paired.sort_values("object_ratio").reset_index(drop=True)


def ecdf(values):
    values = np.sort(np.asarray(values, dtype=float))
    if values.size == 0:
        return np.array([]), np.array([])
    cumulative = np.arange(1, values.size + 1) / values.size
    return values, cumulative


def add_quantile_guides(ax, values, color, linestyle, label_side):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return

    quantile_specs = [
        (0.10, 0.015, "bottom"),
        (0.50, 0.015, "bottom"),
        (0.90, -0.015, "top"),
    ]
    x_offset = 0.02 if label_side == "right" else -0.02
    x_align = "left" if label_side == "right" else "right"

    for q, y_offset, va in quantile_specs:
        x_q = float(np.quantile(values, q))
        y_q = q
        ax.hlines(
            y=y_q,
            xmin=0.0,
            xmax=x_q,
            color=color,
            linestyle=linestyle,
            linewidth=1.0,
            alpha=0.22,
            zorder=1,
        )
        ax.vlines(
            x=x_q,
            ymin=0.0,
            ymax=y_q,
            color=color,
            linestyle=linestyle,
            linewidth=1.0,
            alpha=0.22,
            zorder=1,
        )
        ax.text(
            np.clip(x_q + x_offset, 0.02, 0.98),
            np.clip(y_q + y_offset, 0.02, 0.98),
            f"{x_q:.2f}",
            fontsize=QUANTILE_LABEL_SIZE,
            ha=x_align,
            va=va,
            color=color,
            clip_on=False,
        )


result_order = ["safe", "unknown", "unsafe"]
outcome_styles = {
    "safe": {"title": "Safe", "color": "#111111"},
    "unknown": {"title": "Unknown", "color": "#7A7A7A"},
    "unsafe": {"title": "Unsafe", "color": "#B22222"},
}

BASELINE_LABEL = r"$\alpha\beta$-CROWN"
CSI_LABEL = r"$\alpha\beta$-CROWN w/ CSI [object]"

method_styles = {
    BASELINE_LABEL: {"linestyle": "--", "linewidth": 3.0},
    CSI_LABEL: {"linestyle": "-", "linewidth": 3.0},
}

ratio_groups = {
    CSI_LABEL: {
        label: paired.loc[paired["obj_result"] == label, "object_ratio"].astype(float).to_numpy()
        for label in result_order
    },
    BASELINE_LABEL: {
        label: paired.loc[paired["glb_result"] == label, "object_ratio"].astype(float).to_numpy()
        for label in result_order
    },
}

eps_values = paired["eps"].dropna().unique()
eps_text = rf", $\epsilon={eps_values[0]:g}$" if len(eps_values) == 1 else ""

fig, axes = plt.subplots(
    1,
    3,
    figsize=DEFAULT_FIGSIZE,
    sharex=True,
    sharey=True,
)

for ax, label in zip(axes, result_order):
    outcome_style = outcome_styles[label]
    for method, style in method_styles.items():
        x_ecdf, y_ecdf = ecdf(ratio_groups[method][label])
        if len(x_ecdf) == 0:
            continue
        ax.plot(
            x_ecdf,
            y_ecdf,
            color=outcome_style["color"],
            linestyle=style["linestyle"],
            linewidth=style["linewidth"],
            marker="",
            markersize=0,
            solid_capstyle="butt",
            dash_capstyle="butt",
            label=method,
        )
        add_quantile_guides(
            ax,
            ratio_groups[method][label],
            color=outcome_style["color"],
            linestyle=style["linestyle"],
            label_side="left" if method == BASELINE_LABEL else "right",
        )

    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)
    ax.set_xticks(np.linspace(0.0, 1.0, 6))
    ax.set_yticks(np.linspace(0.0, 1.0, 6))
    ax.set_title(
        f"{outcome_style['title']} --- ImageNet",
        fontsize=AXIS_TITLE_SIZE,
        pad=10,
        color=outcome_style["color"],
    )
    ax.tick_params(axis="both", labelsize=TICK_LABEL_SIZE)
    ax.grid(axis="y", linestyle="--", alpha=0.28)
    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)

fig.supxlabel("Object ratio", y=0.05, fontsize=AXIS_LABEL_SIZE)
fig.supylabel("Cumulative fraction of images", x=0.04, fontsize=AXIS_LABEL_SIZE)

handles = [
    mpl.lines.Line2D(
        [0], [0],
        color="#111111",
        linewidth=style["linewidth"],
        linestyle=style["linestyle"],
        marker="",
        markersize=0,
        solid_capstyle="butt",
        dash_capstyle="butt",
        label=method,
    )
    for method, style in method_styles.items()
]

axes[1].legend(
    handles=handles,
    loc="upper left",
    bbox_to_anchor=(0.02, 0.98),
    ncol=1,
    frameon=False,
    fontsize=LEGEND_FONT_SIZE,
    handlelength=2.6,
    columnspacing=1.8,
    borderaxespad=0.0,
)

fig.subplots_adjust(top=0.80, bottom=0.18, left=0.08, right=0.98, wspace=0.14)

print(f"Using CSV: {CSV_PATH}")
print(f"Paired rows: {len(paired)}")

for method in [BASELINE_LABEL, CSI_LABEL]:
    counts = {label: int(len(ratio_groups[method][label])) for label in result_order}
    print(f"{method}: {counts}")

# Optional save:
pdf_path = PLOT_DIR / "object_ratio_imagenet_cumulative.pdf"
fig.savefig(pdf_path, bbox_inches="tight")
print(f"Saved: {pdf_path}")

plt.show()

Using CSV: /Users/zd3504phd/Desktop/XAIV/results/vggnet16/all_k_vggnet16.csv
Paired rows: 181
$\alpha\beta$-CROWN: {'safe': 13, 'unknown': 72, 'unsafe': 96}
$\alpha\beta$-CROWN w/ CSI [object]: {'safe': 58, 'unknown': 28, 'unsafe': 95}


Saved: /Users/zd3504phd/Desktop/XAIV/plots/5.4.2_ratio/object_ratio_imagenet_cumulative.pdf


/var/folders/fr/3yg6c3f55w58dtlsnjcqjmth0000gq/T/ipykernel_73163/3431428530.py:375: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
